# Imports

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.tree import DecisionTreeClassifier

from core.dataset import TimeSeriesDataset
from core.evaluation import Evaluation
from core.feature_transformer import C22Feature, FeatureCategory, FeatureTransformer
from core.hvac_data_gen import HVACDataGenerator
from core.models import Catch22MPModel, EuclideanDistModel, FeatureWeighter, IForestModel, UniformWeighter
from core.viz import plot_cases

/Users/shelleygoel/miniconda3/envs/TSB-AD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load HVAC Data

In [2]:
regenerate_data = False 
if regenerate_data:
    # Generate HVAC Data with longer history
    generator = HVACDataGenerator(seed=10)
    hvac_df = generator.generate_dataset(
        num_containers=1000,
        start_time=datetime(2026, 1, 15),
        duration_days=10,
    )
    hvac_df.to_parquet(Path("../datasets/hvac_anomalies_v04.parquet"))
else:
    hvac_df = pd.read_parquet(Path("../datasets/hvac_anomalies_v042026.parquet"))

    hvac_df = hvac_df.copy()

hvac_df["TmpRet"] = hvac_df.groupby(["container_id", "unit"])["TmpRet"].transform(
        lambda x: x.rolling(window=10, min_periods=1).mean()
    )

# Wrap in TimeSeriesDataset
col_map = {
    "entity": "container_id",
    "time": "timestamp_et",
    "value_cols": ["TmpRet"],
    "label": "anomaly",
    "label_type": "anomaly_type",
    "sub_entity": "unit",
}
hvac_ds = TimeSeriesDataset(hvac_df, col_map)
print(hvac_ds.anomaly_summary())

sample_for_exp = False
if sample_for_exp:
    n_sample_size = 100
    sampled_entities = hvac_ds.sample_entities(n_cases=50, label_type="frequency", random_state=42)
    normal_entities = hvac_ds.sample_entities(n_cases=n_sample_size, label_type="normal", random_state=42)

    train_entities = np.concatenate([sampled_entities, normal_entities])
    hvac_ds = TimeSeriesDataset(hvac_df[hvac_df["container_id"].isin(train_entities)], col_map)
    print(hvac_ds.anomaly_summary())
print(hvac_ds.anomaly_summary())

  label_type  entity_count
0     normal           798
1  amplitude            72
2  frequency            67
3        lag            63
  label_type  entity_count
0     normal           798
1  amplitude            72
2  frequency            67
3        lag            63


## Evaluation Harness Instantiation
- should be instantiated once and used for all the models- that way labels are computed once and cached - more computationally efficient

In [3]:
ev = Evaluation(level="day")


# Euclidean Distance Model


In [4]:

eucl = EuclideanDistModel(feature_col="TmpRet", smooth_window=1, dist_window=12 * 60, strategy="iqr")
eucl_day_scores = eucl.score_anomalies(hvac_ds, level="day")

# Catch22 MP Model

## C22 Features Calculation

In [5]:
# 3. Feature transform
feats_to_calc = [
    C22Feature.CO_f1ecac,
    C22Feature.CO_FirstMin_ac,
    C22Feature.IN_AutoMutualInfoStats_40_gaussian_fmmi,
    C22Feature.SP_Summaries_welch_rect_area_5_1,
    C22Feature.SP_Summaries_welch_rect_centroid,
]
ft = FeatureTransformer(
    raw_data_columns=["TmpRet"], window_size=12 * 60, stride=60, n_jobs=8, c22_features=feats_to_calc
)
feat_ds = ft.transform(hvac_ds)  # categories=[FeatureCategory.C22_RAW_DIFF])

FeatureTransformer: 100%|██████████| 1000/1000 [01:14<00:00, 13.49it/s]


In [6]:
feat_ds.df.shape

(229000, 47)

### Visualize an example

In [7]:
plot_example = False 
if plot_example:
    calc_features = sum([list(v) for v in ft.feature_map.values()], [])
    wanted_feats = [
        "SP_Summaries_welch_rect_centroid",
        "CO_f1ecac",
        "IN_AutoMutualInfoStats_40_gaussian_fmmi",
        "PD_PeriodicityWang_th0.01",
    ]
    feat_cfg = feat_ds.to_plot_cfg()
    feat_cfg.value_cols = [
        col for col in calc_features
        if "_featdiff" in col and any(f in col for f in wanted_feats)
    ]
    figs = plot_cases(
        [hvac_ds.to_plot_cfg(),feat_cfg],
        sample_from=hvac_ds,
        n_cases=1,
        # label_type='frequency',
        label_type='lag',
        labels_from=hvac_ds.to_plot_cfg(),
        random_state=42,
    )


    ckpt_dir = Path(f"checkpoints/hvac/")
    ckpt_dir.mkdir(exist_ok=True)
        
    figs[0].write_html(ckpt_dir / f"example_case_c22_lag.html")

## C22MP: Score Anomalies
- Left C22MP Profile Calculation
- day Level Scores

In [8]:
calc_features = sum([list(v) for v in ft.feature_map.values()], [])

class CustomWeighter(FeatureWeighter):
    def compute_weights(self, preferred_features: list) -> dict:
        return dict([(feat, 1) for feat in preferred_features])


# pattern = re.compile(
#     r"(?=.*_featdiff)(?=.*(SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid|PD_PeriodicityWang_th0.01|CO_f1ecac|CO_FirstMin_ac|IN_AutoMutualInfoStats_40_gaussian_fmmi))"
# )
# pattern = re.compile(
#     r"(?=.*_featdiff)(?=.*(SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid|CO_f1ecac))" 
# )
# pattern = re.compile(
#     r"^(?!.*_featdiff).*(SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid|CO_f1ecac)"
# )
# pattern = re.compile(r'(?=.*_featdiff)(?=.*(CO_f1ecac))')
# pattern = re.compile(r"^(?!.*_featdiff).*(0_1__CO_f1ecac|0_2__CO_f1ecac|1_2__CO_f1ecac)")
# pattern = re.compile(r'SP_Summaries_welch_rect_area_5_1|SP_Summaries_welch_rect_centroid')

cw = CustomWeighter()
pref_feat_exp = [
    ("C22+MP Raw Feat", ["TmpRet__0__CO_f1ecac", "TmpRet__1__CO_f1ecac", "TmpRet__2__CO_f1ecac"]),
    ("C22+MP Feat Diff",
        [
            "TmpRet__featdiff_0_1__CO_f1ecac",
            "TmpRet__featdiff_0_2__CO_f1ecac",
            "TmpRet__featdiff_1_2__CO_f1ecac",
        ],
    ),
    ("C22+MP Raw Diff Feat", ["TmpRet__0_1__CO_f1ecac", "TmpRet__0_2__CO_f1ecac", "TmpRet__1_2__CO_f1ecac"]),
]
stride = ft.stride  # 45
exclude_zone = 1440 // stride  # 32
print(f"{stride=}, {exclude_zone=}")
c22_mps = {}
c22_day_scores = {}
for model_name, pref_features in pref_feat_exp:
    custom_wei = cw.compute_weights(preferred_features=pref_features)
    # 4. Fit MP once
    c22mp = Catch22MPModel(
        exclude_zone=exclude_zone,
        early_abandon=False,  # brute force — fast enough
    )
    mp_ds, debug = c22mp.fit_profile(feat_ds, weights=custom_wei)
    scores = c22mp.score(mp_ds, level="day", day_agg_stat="p90")
    c22_mps[model_name] = mp_ds
    c22_day_scores[model_name] = scores


stride=60, exclude_zone=24


Catch22MP fit_profile:   0%|          | 0/1000 [00:00<?, ?it/s]

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
Catch22MP fit_profile: 100%|██████████| 1000/1000 [00:00<00:00, 1788.73it/s]


In [9]:
scores_long = pd.concat(
    [s.df[["anomaly_score"]].dropna().assign(model=name) for name, s in c22_day_scores.items()],
    ignore_index=True,
)
px.histogram(scores_long, x="anomaly_score", color="model", barmode="overlay", opacity=0.5)



**Observations **
- the score distributions are left skewed as expected so this is a good sanity check

# Isolation Forest Model

In [10]:
iforest_day_scores = {} 
hand_selected_feats = [
            "TmpRet__featdiff_0_1__CO_f1ecac",
            "TmpRet__featdiff_0_2__CO_f1ecac",
            "TmpRet__featdiff_1_2__CO_f1ecac",
        ]

iforest = IForestModel(fit_scope="global", contamination=0.02, feature_cols=hand_selected_feats)
iforest_day_scores["IForest+C22 Feat"] = iforest.score_anomalies(feat_ds, level="day", day_agg_stat="p90")
px.histogram(iforest_day_scores["IForest+C22 Feat"].df["anomaly_score"])

# Compare Models: PR curves

In [11]:
fig = ev.plot_pr_curves_compared(
    # {"Catch22MP": c22_day_scores},
    {**c22_day_scores, "Euclidean Distance": eucl_day_scores, **iforest_day_scores},
    hvac_ds,
    label_types=["frequency", "lag"]
)
fig.update_layout(width=1200)
fig.show()

In [12]:
ev.compare(
    # {"Catch22MP": c22_day_scores},
    {**c22_day_scores, "Euclidean Distance": eucl_day_scores, **iforest_day_scores},
    hvac_ds,
    label_types=["frequency", "lag"]
)

C22+MP Raw Feat  C22+MP Feat Diff  C22+MP Raw Diff Feat  \
label_type metric                                                             
frequency  auc_pr          0.510617          0.690156              0.406537   
           auc_roc         0.964535          0.963881              0.942949   
lag        auc_pr          0.225261          0.465397              0.308295   
           auc_roc         0.925757          0.918241              0.922157   
overall    auc_pr          0.511915          0.692069              0.492699   
           auc_roc         0.944755          0.940600              0.932343   

                    Euclidean Distance  IForest+C22 Feat  
label_type metric                                         
frequency  auc_pr             0.838824          0.935489  
           auc_roc            0.945365          0.987195  
lag        auc_pr             0.874093          0.799381  
           auc_roc            0.977438          0.971382  
overall    auc_pr             0.892869          0.917434  
           auc_roc            0.961479          0.979250

# Feature Selection using Classification
- using a sampled of labeled data
- timestamp labels aggregated to window level label
  - Window is labeled as anomaly if even a single timestamp is anomalous - since we want to have high recall.
- fit a decision tree classifier - using balanced accuracy as measure as anomalies are rate
- to find features which can separate the normal from anomalous cases
- Selected Features can be used in any model - MP or Iforest

## Sample Cases for classifier

In [13]:
sampled_entities = hvac_ds.sample_entities(n_cases=10, label_type="frequency", random_state=42)
normal_entities = hvac_ds.sample_entities(n_cases=100, label_type="normal", random_state=42) 

train_entities = np.concatenate([sampled_entities, normal_entities])

entity_col = feat_ds.col_map["entity"]

feat_ds_train = TimeSeriesDataset(
    feat_ds.df[feat_ds.df[entity_col].isin(train_entities)].copy(),
    feat_ds.col_map,
)
ds_train = TimeSeriesDataset(hvac_df[hvac_df["container_id"].isin(train_entities)], col_map)

window_size = ft.window_size
stride = ft.stride
entity_col = hvac_ds.col_map["entity"]
time_col = hvac_ds.col_map["time"]
label_col = hvac_ds.col_map["label"]

ts_labels_df = ds_train.ts_labels()  # entity, time, label, [label_type]

# --- 1. Label each feature window ---
# Treat it as a "feature on the label": max(label) over [t, t+window_size),
# strided by ft.stride. Vectorized via pivot to (time, entity) then numpy
# strided indexing — mirrors FeatureTransformer's windowing.
label_mat = (
    ts_labels_df.pivot(index=time_col, columns=entity_col, values=label_col)
    .sort_index().astype(int)
)
n_windows = (len(label_mat) - window_size) // stride + 1
starts = np.arange(n_windows) * stride
window_idx = starts[:, None] + np.arange(window_size)[None, :]  # (n_windows, W)

window_label_mat = label_mat.values[window_idx].max(axis=1)  # (n_windows, n_entities)
window_labels_df = (
    pd.DataFrame(window_label_mat, index=label_mat.index.values[starts], columns=label_mat.columns)
    .rename_axis(time_col).reset_index()
    .melt(id_vars=time_col, var_name=entity_col, value_name="window_label")
)

# --- 2. Build (X, y) ---
merged = feat_ds_train.df.merge(window_labels_df, on=[entity_col, time_col])

feature_cols = feat_ds_train.col_map["value_cols"]
# for frequency anomaly we know that 
feature_cols = ft.feature_map[FeatureCategory.C22_FEAT_DIFF]
X = merged[feature_cols].values

y = merged["window_label"].astype(int).values


In [14]:

import numpy as np                                                                                                                                                                                                                                                                                                
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

def per_feature_auc_roc(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    y = np.asarray(y).astype(bool)
    out = np.full(X.shape[1], np.nan)
    for j in range(X.shape[1]):
        col = X[:, j]
        mask = ~np.isnan(col)
        yj = y[mask]
        if yj.any() and not yj.all():
            out[j] = roc_auc_score(yj, col[mask])
    return out

#Direction-agnostic "separability" — useful when you don't care
  # whether high or low values indicate anomaly.
def per_feature_separability(X, y):
    auc = per_feature_auc_roc(X, y)
    return np.maximum(auc, 1 - auc)          # in [0.5, 1.0]

roc_scores = per_feature_separability(X, y)           # (130,)
top_roc_scores = np.argsort(roc_scores)[::-1][:10]               # top 10 features
roc_selected_feat = np.array(feature_cols)[top_roc_scores]
print(roc_selected_feat)                     


['TmpRet__featdiff_0_1__CO_f1ecac' 'TmpRet__featdiff_0_1__CO_FirstMin_ac'
 'TmpRet__featdiff_1_2__CO_f1ecac' 'TmpRet__featdiff_1_2__CO_FirstMin_ac'
 'TmpRet__featdiff_1_2__SP_Summaries_welch_rect_centroid'
 'TmpRet__featdiff_0_1__SP_Summaries_welch_rect_centroid'
 'TmpRet__featdiff_1_2__SP_Summaries_welch_rect_area_5_1'
 'TmpRet__featdiff_1_2__IN_AutoMutualInfoStats_40_gaussian_fmmi'
 'TmpRet__featdiff_0_1__IN_AutoMutualInfoStats_40_gaussian_fmmi'
 'TmpRet__featdiff_0_1__SP_Summaries_welch_rect_area_5_1']


In [15]:


print(f"n_windows={len(y)}, n_anomaly_windows={y.sum()}, base_rate={y.mean():.3f}")

# --- 3. Fit Decision Tree + rank features ---
clf = DecisionTreeClassifier(
    max_depth=10,
    class_weight="balanced",  # anomalies are rare — rebalance
    random_state=42,
)
clf.fit(X, y)

dt_feat_imp = (
    pd.DataFrame({"feature": feature_cols, "importance": clf.feature_importances_})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
dt_selected_feat = dt_feat_imp.head(10)["feature"].values
dt_selected_feat

n_windows=25190, n_anomaly_windows=589, base_rate=0.023


array(['TmpRet__featdiff_1_2__CO_FirstMin_ac',
       'TmpRet__featdiff_0_1__CO_f1ecac',
       'TmpRet__featdiff_1_2__CO_f1ecac',
       'TmpRet__featdiff_0_2__CO_f1ecac',
       'TmpRet__featdiff_0_2__CO_FirstMin_ac',
       'TmpRet__featdiff_1_2__SP_Summaries_welch_rect_area_5_1',
       'TmpRet__featdiff_0_1__SP_Summaries_welch_rect_area_5_1',
       'TmpRet__featdiff_0_2__SP_Summaries_welch_rect_area_5_1',
       'TmpRet__featdiff_0_1__CO_FirstMin_ac',
       'TmpRet__featdiff_1_2__SP_Summaries_welch_rect_centroid'],
      dtype=object)

In [16]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    "feature": pd.Series(feature_cols).str.replace("TmpRet__", "", regex=False),
    "DT importance": clf.feature_importances_,
    "AUC-ROC": roc_scores,
}).melt(id_vars="feature", var_name="metric", value_name="score")

order = (
    df[df["metric"] == "AUC-ROC"]
    .sort_values("score", ascending=False)["feature"].tolist()
)

fig = px.bar(
    df, x="feature", y="score",
    facet_row="metric",
    category_orders={"feature": order},
    title="Per-feature importance: DT vs AUC-ROC",
)
# fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(xaxis_tickangle=-45, height=600)
fig.show()

# IForest on Auto-Selected Features

In [17]:
exp = [
    ("IForest+C22 ROC Feat(5)", roc_selected_feat[:5]),
    ("IForest+C22 ROC Feat(3)", roc_selected_feat[:3]),
    ("IForest+C22 DT Feat", dt_selected_feat[:3])

]
for md, feat in exp:
    iforest = IForestModel(fit_scope="global", contamination=0.02, feature_cols=feat)
    iforest_day_scores[md] = iforest.score_anomalies(feat_ds, level="day", day_agg_stat="p90")
    fig = px.histogram(iforest_day_scores[md].df["anomaly_score"])
    fig.update_layout(height=200, width=500)
    fig.show()


## PR Curve

In [18]:

fig = ev.plot_pr_curves_compared(
    # {k: c22_day_scores[k] for k in ["feat_diff", "clf_feat_select"]} | {"iforest_clf_select": iforest_day_scores_v2, "iforest_hand_select": iforest_day_scores_v1, "iforest_v3": iforest_day_scores_v3},
    iforest_day_scores,
    hvac_ds,
    label_types=["frequency", "lag"]
)
fig.show()

# PR Curves: Selected Model

In [19]:
print(c22_day_scores.keys(), iforest_day_scores.keys())

dict_keys(['C22+MP Raw Feat', 'C22+MP Feat Diff', 'C22+MP Raw Diff Feat']) dict_keys(['IForest+C22 Feat', 'IForest+C22 ROC Feat(5)', 'IForest+C22 ROC Feat(3)', 'IForest+C22 DT Feat'])


In [20]:
models_to_compare = {
    'Catch22+MP': c22_day_scores["C22+MP Feat Diff"],
    'IForest+C22+Feat Selection': iforest_day_scores["IForest+C22 ROC Feat(3)"], 
    'Custom Model: Eucl Distance': eucl_day_scores
}
                     
iforest_models = []
fig = ev.plot_pr_curves_compared(
    models_to_compare,
    hvac_ds,
    label_types=["frequency", "lag", "amplitude"]
)
fig.update_layout(width=1500)
fig.show()
ev.compare(
    models_to_compare,
    hvac_ds,
    label_types=["frequency"]
    
)

Catch22+MP  IForest+C22+Feat Selection  \
label_type metric                                            
frequency  auc_pr     0.690156                    0.880345   
           auc_roc    0.963881                    0.983510   
overall    auc_pr     0.690156                    0.880345   
           auc_roc    0.963881                    0.983510   

                    Custom Model: Eucl Distance  
label_type metric                                
frequency  auc_pr                      0.838824  
           auc_roc                     0.945365  
overall    auc_pr                      0.838824  
           auc_roc                     0.945365

*Observations*
Hand selected features beat auto-selected features, but they are still among top features of auto-selected set. 
- Can try different subsets of top 5 features. 

# Debugging

# Latent Failure Detection: Amplitude Anomaly

In [32]:
mname = "C22+MP Feat Diff"
c22_model_scores = c22_day_scores[mname]
mp_ds = c22_mps[mname]
c22_mp_feats = [
            "TmpRet__featdiff_0_1__CO_f1ecac",
            "TmpRet__featdiff_0_2__CO_f1ecac",
            "TmpRet__featdiff_1_2__CO_f1ecac",
        ]
        
threshold = 0.7
iforest_model_scores = iforest_day_scores["IForest+C22 ROC Feat(5)"]
tp  = ev.true_positives(iforest_model_scores, hvac_ds, threshold=threshold, top_k=5, label_type="lag")
tp

,container_id,day,anomaly_score,anomaly_type
0,608,2026-01-21,0.824071,lag
1,608,2026-01-23,0.796141,lag
2,789,2026-01-18,0.781479,lag
3,209,2026-01-22,0.774075,lag
4,490,2026-01-21,0.766573,lag


In [33]:

feat_cfg = feat_ds.to_plot_cfg()
feat_cfg.value_cols = roc_selected_feat[:5] 
skip = False
lbt = "unknown"
if not skip:
    figs = plot_cases(
        [hvac_ds.to_plot_cfg(), iforest_model_scores.to_plot_cfg(), feat_cfg,],
        sample_from=hvac_ds,
        # entity_ids=false_positives["container_id"].unique(),
        entity_ids=tp["container_id"].unique(),
        n_cases=3,
        label_type="amplitude",
        labels_from=hvac_ds.to_plot_cfg(),
        random_state=42,
    )


ckpt_dir = Path(f"checkpoints/hvac/{lbt}/")
ckpt_dir.mkdir(exist_ok=True)

for i, fig in enumerate(figs):
    
    fig.write_html(ckpt_dir / f"case_{i}.html")


# Appendix

### Features for frequency
Direct spectral (Welch power spectrum)
  - SP_Summaries_welch_rect_area_5_1 — power in the lowest 1/5 of the spectrum. Shifts when dominant frequency moves.
  - SP_Summaries_welch_rect_centroid — frequency at which power is concentrated. Clearest signal for "wrong period."

  Periodicity detectors
  - PD_PeriodicityWang_th0.01 — Wang's dominant-periodicity estimate. Directly encodes the cycle length.
  - CO_f1ecac — first crossing of ACF at 1/e. A characteristic timescale — changes with period.
  - CO_FirstMin_ac — first minimum of the autocorrelation function. Roughly half the dominant period.
  - IN_AutoMutualInfoStats_40_gaussian_fmmi — first minimum of Auto-Mutual-Information. Nonlinear analog of CO_FirstMin_ac.

  Scaling / long-range (weaker, indirect)
  - SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1 — DFA scaling exponent.
  - SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1 — R/S range fit. Both reflect how power distributes across scales.

  Residual-autocorrelation
  - FC_LocalSimple_mean1_tauresrat — ratio of residual ACF timescale to raw ACF timescale after a simple forecast. Sensitive to how "learnable" the periodic structure is.